# QLoRA for Biomedical POS Tagging

## Overview

This notebook demonstrates **QLoRA** (Quantized Low-Rank Adaptation) for **Biomedical Part-of-Speech (POS) Tagging** using a small, pre-quantized causal LLM (**Qwen3-4B-Instruct** in 4-bit via Unsloth).

The notebook is structured in three acts:

| # | Approach | What we learn |
|---|---|---|
| 1 | **Simple instruction prompting** | Baseline: how well does the raw, quantized model follow a tag-this-sentence instruction? |
| 2 | **Chain-of-Thought (CoT) prompting** | Does step-by-step reasoning improve tagging without any weight updates? |
| 3 | **QLoRA fine-tuning** | Fine-tune only ~0.5 % of parameters on a biomedical POS corpus; compare to the two baselines. |

### Why Biomedical POS?

Biomedical text presents unique challenges for standard POS taggers trained on general-domain corpora:

*   **Complex nominals** — multi-word enzyme names, gene symbols, and chemical formulae that span several tokens (e.g., *"phosphatidylinositol 3-kinase"*).
*   **Unconventional capitalisation** — gene/protein names are often mid-sentence proper nouns, yet standard taggers miss them.
*   **High noun density** — biomedical sentences carry far more consecutive nouns and noun-modifying adjectives than news or web text, compressing ambiguity into tight windows.
*   **Domain-specific abbreviations** — acronyms like *"IL-6"*, *"mRNA"*, or *"bp"* do not appear in general treebanks and confuse standard taggers.

Accurate POS tagging is a prerequisite for downstream tasks such as named-entity recognition (NER), relation extraction, and clinical information retrieval.

### Model

We use `unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit`, an optimised 4-bit quantized checkpoint of Qwen3-4B-Instruct. QLoRA (Dettmers et al., 2023) then injects small trainable LoRA adapters into the frozen quantized base, keeping GPU memory well under 12 GB for a 4B model.

## Setup

### Dependencies

!pip install -q -U \
    transformers datasets peft evaluate seqeval accelerate \
    bitsandbytes scipy scikit-learn trl unsloth

### Imports & Device Check

In [5]:
import re, json, gc, warnings
import numpy as np
import torch
import transformers
import datasets
import evaluate
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
from functools import partial

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Using device: cuda
  GPU: NVIDIA GeForce RTX 4060 Ti
  VRAM: 8.2 GB


### Load the model

In [ ]:
MODEL_ID = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",           # spread across available GPUs / CPU offload
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: { trainable_params:,}")
print(f"  → {100*trainable_params/total_params:.2f}% trainable (pre-LoRA)")


## Dataset — CRAFT Biomedical Treebank

### What is CRAFT?

The **Colorado Richly Annotated Full-Text (CRAFT)** corpus is a gold-standard, manually
annotated biomedical NLP resource built at the University of Colorado. It consists of
67 full-text articles from the PubMed Central Open Access subset, all drawn from the
domain of mouse genomics and molecular biology.

The corpus is richly layered: the same articles carry annotations for part-of-speech,
syntactic constituency parses, coreference, named entities (genes, proteins, chemicals,
species, GO terms, etc.), and semantic roles. For this notebook we use the
**POS / Universal Dependencies** layer.

**HuggingFace identifier:** `nlpie/craft-treebank`
**Paper:** Bada et al. (2012), *"Concept annotation in the CRAFT corpus"*, BMC Bioinformatics.
**Licence:** Creative Commons Attribution 3.0 (CC BY 3.0).

---

### Tag Set

CRAFT POS annotations follow the **Penn Treebank (PTB) tagset**, which is the standard
for English biomedical NLP. The table below lists every tag you will encounter, with
biomedical-domain examples:

| Tag | Full name | Biomedical example |
|-----|-----------|--------------------|
| `NN` | Noun, singular or mass | *kinase*, *pathway*, *mRNA* |
| `NNS` | Noun, plural | *receptors*, *alleles*, *mice* |
| `NNP` | Proper noun, singular | *Mus*, *NCBI*, *Sox9* |
| `NNPS` | Proper noun, plural | *Sprague-Dawleys* |
| `VB` | Verb, base form | *bind*, *activate*, *express* |
| `VBD` | Verb, past tense | *increased*, *inhibited* |
| `VBG` | Verb, gerund/present participle | *encoding*, *regulating* |
| `VBN` | Verb, past participle | *expressed*, *phosphorylated* |
| `VBP` | Verb, non-3rd person singular present | *regulate*, *interact* |
| `VBZ` | Verb, 3rd person singular present | *encodes*, *binds* |
| `JJ` | Adjective | *murine*, *genomic*, *cystic* |
| `JJR` | Adjective, comparative | *higher*, *smaller* |
| `JJS` | Adjective, superlative | *highest*, *strongest* |
| `RB` | Adverb | *significantly*, *previously* |
| `RBR` | Adverb, comparative | *more*, *further* |
| `RBS` | Adverb, superlative | *most*, *least* |
| `DT` | Determiner | *the*, *a*, *this* |
| `PDT` | Predeterminer | *all*, *both*, *half* |
| `IN` | Preposition / subordinating conjunction | *of*, *in*, *that* |
| `CC` | Coordinating conjunction | *and*, *or*, *but* |
| `PRP` | Personal pronoun | *it*, *they*, *we* |
| `PRP$` | Possessive pronoun | *its*, *their* |
| `WP` | Wh-pronoun | *which*, *that* |
| `WDT` | Wh-determiner | *which*, *that* |
| `WRB` | Wh-adverb | *where*, *when* |
| `CD` | Cardinal number | *67*, *three*, *1.5* |
| `EX` | Existential *there* | *there* (as in "there are") |
| `FW` | Foreign word | *in vitro*, *de novo* |
| `LS` | List item marker | *1.*, *A)* |
| `MD` | Modal | *can*, *may*, *should* |
| `POS` | Possessive ending | *'s* |
| `RP` | Particle | *up* (in "regulate up") |
| `SYM` | Symbol | *%*, *+*, *α* |
| `TO` | *to* | *to* (infinitive marker) |
| `UH` | Interjection | *(rare in biomedical text)* |
| `-LRB-` | Left bracket | *(*, *[* |
| `-RRB-` | Right bracket | *)*, *]* |
| `''` | Closing quotation mark | *"* |
| ` `` ` | Opening quotation mark | *"* |
| `,` | Comma | *,* |
| `.` | Sentence-final punctuation | *.*, *?*, *!* |
| `:` | Mid-sentence punctuation | *:*, *;*, *—* |
| `$` | Dollar sign | *(rare; may appear in numeric ranges)* |
| `#` | Pound sign | *#* |
| `HYPH` | Hyphen | *-* (word-internal) |
| `NFP` | Superfluous punctuation | *...*, repeated dashes |
| `XX` | Unknown / other | *(catch-all for unrecognised tokens)* |

---

### Representative Examples

```
Sentence: The Fgfr3 gene encodes a receptor tyrosine kinase .
Tags:     DT  NNP  NN   VBZ  DT  NN       NN       NN      .

Sentence: These results suggest that Akt phosphorylates FOXO1 in vivo .
Tags:     DT    NNS    VBP     IN   NNP VBZ             NNP   IN NN   .

Sentence: Homozygous null mutants die shortly after birth .
Tags:     JJ         JJ   NNS     VBP IN     RB    IN    NN .
```

Notice how gene symbols (`Fgfr3`, `Akt`, `FOXO1`) are tagged `NNP` (proper noun),
Latin phrases (*"in vivo"*) split into `IN NN`, and biomedical nominals stack
multiple `NN` tokens consecutively — all patterns absent or rare in general-domain text.

In [ ]:
# ── Load CRAFT Treebank ───────────────────────────────────────────────────────
raw = datasets.load_dataset("nlpie/craft-treebank", trust_remote_code=True)
print(raw)
print()
print("Sample training example:")
print(raw["train"][0])

In [4]:
# ── Inspect PTB tag names ─────────────────────────────────────────────────────
# CRAFT stores POS as integer ids; decode them from the ClassLabel feature
pos_feature = raw["train"].features["pos_tags"]
id2pos = {i: pos_feature.feature.int2str(i) for i in range(pos_feature.feature.num_classes)}
pos2id = {v: k for k, v in id2pos.items()}

print(f"Number of PTB tags: {len(id2pos)}")
print()
print("Tag mapping:")
for k, v in sorted(id2pos.items()):
    print(f"  {k:3d} → {v}")

Generating train split: 100%|██████████| 12544/12544 [00:00<00:00, 607600.58 examples/s]

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})

Sample training example:
{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.', 'comments': ['newdoc id = weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000', '__SENT_ID__', 'newpar id = weblo

In [ ]:
# Distribution of sentence lengths
import collections
lens = [len(ex["tokens"]) for ex in train_ds]
print(f"Sentence length — min: {min(lens)}, max: {max(lens)}, mean: {np.mean(lens):.1f}")

# Most frequent tags
tag_counts = collections.Counter(
    id2pos[t] for ex in train_ds for t in ex["pos_tags"]
)
print("\nTop-15 tags in training set:")
for tag, cnt in tag_counts.most_common(15):
    print(f"  {tag:<8} {cnt:5d}")

In [ ]:
# ── Build a plain-text helper (tokens → tagged string) ───────────────────────
def example_to_str(example):
    """Return a single-line annotated string for display / prompting."""
    pairs = zip(example["tokens"], example["pos_tags"])
    return " ".join(f"{tok}/{id2pos[tag]}" for tok, tag in pairs)

# Quick sanity check — print a few annotated sentences
for i in range(3):
    print(f"[{i}] {example_to_str(raw['train'][i])}")
    print()

In [ ]:
# ── Keep a small, balanced split to speed up the demo ────────────────────────
TRAIN_SIZE = 1000
EVAL_SIZE  = 200
TEST_SIZE  = 200

train_ds = raw["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
eval_ds  = raw["validation"].shuffle(seed=42).select(range(EVAL_SIZE))
test_ds  = raw["test"].shuffle(seed=42).select(range(TEST_SIZE))

print(f"Train: {len(train_ds)} | Val: {len(eval_ds)} | Test: {len(test_ds)}")

## Prompting

We define two prompts:

* **Simple instruction prompt** — direct, imperative instruction with the expected output format.
* **Chain-of-Thought prompt** — asks the model to reason about each word's syntactic role
  before emitting the final tag sequence.


In [ ]:
PTB_LIST = ", ".join(sorted(set(id2pos.values())))

def build_instruction_prompt(tokens: list[str]) -> str:
    """Simple instruction prompt for biomedical POS tagging."""
    sentence = " ".join(tokens)
    system = (
        "You are a linguistic expert specialising in biomedical text. "
        "Your task is to assign Penn Treebank (PTB) POS tags to each token in the given sentence. "
        f"The valid PTB tags are: {PTB_LIST}. "
        "Return ONLY a JSON array of objects with keys 'token' and 'tag', "
        "with no additional commentary."
    )
    user = f"Sentence: {sentence}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

def build_cot_prompt(tokens: list[str]) -> str:
    """Chain-of-Thought prompt for biomedical POS tagging."""
    sentence = " ".join(tokens)
    system = (
        "You are a linguistic expert specialising in biomedical text. "
        "Your task is to assign Penn Treebank (PTB) POS tags to each token in the given sentence. "
        f"The valid PTB tags are: {PTB_LIST}. "
        "First, briefly reason about each word's grammatical role (one line per token), "
        "paying special attention to gene names, protein names, Latin phrases, and "
        "domain-specific abbreviations. "
        "Then, after the reasoning, output a JSON array of objects with keys 'token' and 'tag' "
        "inside a ```json ... ``` code block. Do not add any text after the code block."
    )
    user = f"Sentence: {sentence}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

# Preview prompts on a sample sentence
sample = test_ds[0]["tokens"]
print("=== INSTRUCTION PROMPT ===")
print(build_instruction_prompt(sample[:8]))
print()
print("=== COT PROMPT ===")
print(build_cot_prompt(sample[:8]))

In [ ]:
def parse_tagged_output(text: str, tokens: list[str]) -> list[str]:
    """
    Extract the list of PTB tags from model output.
    Falls back to 'XX' for unparseable responses.
    """
    # Try to find a JSON array in the output (possibly inside a ```json block)
    json_match = re.search(r'```json\s*(\[.*?\])\s*```', text, re.DOTALL)
    if not json_match:
        json_match = re.search(r'(\[.*?\])', text, re.DOTALL)

    if json_match:
        try:
            parsed = json.loads(json_match.group(1))
            tags = [item.get("tag", "XX").upper() for item in parsed]
            if len(tags) >= len(tokens):
                return tags[:len(tokens)]
            else:
                return tags + ["XX"] * (len(tokens) - len(tags))
        except json.JSONDecodeError:
            pass

    # Last resort: look for space-separated PTB-looking tokens
    valid = set(id2pos.values())
    candidates = [t for t in text.split() if t.upper() in valid]
    if len(candidates) >= len(tokens):
        return candidates[:len(tokens)]

    return ["XX"] * len(tokens)

def generate_tags(prompt: str, tokens: list[str], max_new_tokens: int = 512) -> tuple:
    """Run the model and return (predicted tag list, raw decoded text)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    return parse_tagged_output(text, tokens), text

### Simple instruction prompt
We run the **quantized, un-fine-tuned** model on 50 test sentences
with a plain instruction prompt and measure token-level accuracy and macro-F1.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

EVAL_N = 50   # reduce to speed up; increase for more robust numbers

def evaluate_prompting(build_prompt_fn, dataset, n=EVAL_N, label="Prompting"):
    """Evaluate a prompting strategy on the first n examples of dataset."""
    all_preds, all_golds = [], []

    for i, ex in enumerate(dataset.select(range(n))):
        tokens = ex["tokens"]
        gold   = [id2upos[t] for t in ex["upos"]]
        prompt = build_prompt_fn(tokens)
        preds, raw_text = generate_tags(prompt, tokens)

        all_preds.extend(preds)
        all_golds.extend(gold)

        if i < 2:   # show two examples
            print(f"--- Example {i+1} ---")
            print("Tokens:", tokens)
            print("Gold:  ", gold)
            print("Pred:  ", preds)
            print("Raw output snippet:", raw_text[:300], "...")
            print()

    # Metrics
    
    acc = accuracy_score(all_golds, all_preds)
    f1  = f1_score(all_golds, all_preds, average="macro", zero_division=0)
    print(f"[{label}]  Token Accuracy: {acc:.4f}  |  Macro-F1: {f1:.4f}")
    return acc, f1


acc_instr, f1_instr = evaluate_prompting(
    build_instruction_prompt, test_ds, n=EVAL_N, label="Simple Instruction"
)


### Chain-of-Thought (CoT) Prompting

We repeat the evaluation with the **CoT prompt**, which asks the model to reason
step-by-step before emitting tags.

**Hypothesis:** By forcing the model to verbalise its reasoning about each token's
syntactic role, it anchors on the correct tag category more reliably — especially
for ambiguous legal terms like *"shall"* (AUX vs VERB) or *"party"* (NOUN vs PROPN).


In [ ]:
acc_cot, f1_cot = evaluate_prompting(
    build_cot_prompt, test_ds, n=EVAL_N, label="Chain-of-Thought"
)


In [ ]:
# ── Compare the two baselines ─────────────────────────────────────────────────
print("\n=== Prompting Comparison ===")
print(f"{'Strategy':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 47)
print(f"{'Simple Instruction':<25} {acc_instr:>10.4f} {f1_instr:>10.4f}")
print(f"{'Chain-of-Thought':<25} {acc_cot:>10.4f} {f1_cot:>10.4f}")


## QLoRA Fine-Tuning

### Prepare the Model for QLoRA

In [ ]:
RANK = 16

# Free any previous model state if re-running
gc.collect()
torch.cuda.empty_cache()

# prepare_model_for_kbit_training casts layer norms to fp32
# and enables gradient checkpointing to reduce activation memory
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=RANK,                    # adapter rank
    lora_alpha=2*RANK,           # scaling: alpha/r = 2
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # Target the attention projection layers (Qwen2 naming)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


### Build Supervised Fine-Tuning Dataset

We convert each UD example into a **prompt → completion** pair using the same
instruction prompt format from §5.  The **label** (what we want the model to
generate) is a JSON array of `{token, tag}` objects.

Only the completion tokens contribute to the loss (the prompt tokens are masked
with `-100`), which is standard for instruction-tuning.


In [ ]:
def make_sft_example(example: dict) -> dict:
    """
    Build a tokenised SFT example:
      input_ids = [prompt tokens] + [completion tokens]
      labels    = [-100 ...] + [completion tokens]   (mask the prompt)
    """
    tokens = example["tokens"]
    gold   = [id2upos[t] for t in example["upos"]]
    gold_json = json.dumps([{"token": tok, "tag": tag}
                            for tok, tag in zip(tokens, gold)])

    prompt     = build_instruction_prompt(tokens)
    completion = gold_json + tokenizer.eos_token

    prompt_ids     = tokenizer(prompt,     add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + completion_ids
    labels    = [-100] * len(prompt_ids) + completion_ids

    return {"input_ids": input_ids, "labels": labels,
            "attention_mask": [1] * len(input_ids)}


# Tokenise splits
MAX_LEN = 1024

def tokenise_split(ds):
    ds = ds.map(make_sft_example, remove_columns=ds.column_names)
    # Filter out sequences that exceed the context window
    ds = ds.filter(lambda x: len(x["input_ids"]) <= MAX_LEN)
    return ds

print("Tokenising splits …")
train_tok = tokenise_split(train_ds)
eval_tok  = tokenise_split(eval_ds)
print(f"  Train: {len(train_tok)}, Eval: {len(eval_tok)}")
print(f"  Sample sequence length: {len(train_tok[0]['input_ids'])}")


### Training with the HuggingFace Trainer

In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir="./qlora_legal_pos/",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=False,
    bf16=True,                         # bfloat16 for Ampere+ GPUs
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=20,
    report_to="none",
    optim="paged_adamw_8bit",          # memory-efficient AdamW from bitsandbytes
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

trainer.train()


### Evaluation

After fine-tuning we evaluate with the same `evaluate_prompting` harness as before
(the model still generates JSON; we just parse it the same way).

Because the QLoRA adapters have been trained on the *exact* prompt format and tagset,
we expect substantially higher accuracy than the zero-shot baselines.


In [ ]:
acc_qlora, f1_qlora = evaluate_prompting(
    build_instruction_prompt, test_ds, n=EVAL_N, label="QLoRA"
)


In [ ]:
# ── Final comparison table ────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"  {'Strategy':<28} {'Accuracy':>10} {'Macro-F1':>10}")
print("="*55)
print(f"  {'Simple Instruction (zero-shot)':<28} {acc_instr:>10.4f} {f1_instr:>10.4f}")
print(f"  {'Chain-of-Thought (zero-shot)':<28} {acc_cot:>10.4f} {f1_cot:>10.4f}")
print(f"  {'QLoRA fine-tuned':<28} {acc_qlora:>10.4f} {f1_qlora:>10.4f}")
print("="*55)


In [ ]:
# ── Per-class breakdown (QLoRA) ───────────────────────────────────────────────
from sklearn.metrics import classification_report

all_preds, all_golds = [], []
for ex in test_ds.select(range(EVAL_N)):
    tokens = ex["tokens"]
    gold   = [id2upos[t] for t in ex["upos"]]
    prompt = build_instruction_prompt(tokens)
    preds, _ = generate_tags(prompt, tokens)
    all_preds.extend(preds)
    all_golds.extend(gold)

print(classification_report(all_golds, all_preds, zero_division=0))


### Saving the Adapter

The LoRA adapters are tiny (~60 MB for r=16) and can be saved/loaded independently
of the base model weights, making them easy to share or version-control.

Merging fuses ΔW back into W, producing a standard (un-PEFT) model that requires
no PEFT library at inference time.


In [ ]:
# Save adapter weights only
model.save_pretrained("./qlora_biom_pos_adapter/")
tokenizer.save_pretrained("./qlora_biom_pos_adapter/")
print("Adapter saved.")
